# SACDpy batch multicolor z-stack reconstruction

Pipeline for ONI acquisitions in which every z-plane movie contains sequential 405, 488, 561, and 647 frame blocks. It selects the left camera half for 405/488/561 and the right camera half for 647, reconstructs every channel independently, and writes combined channel-labeled OME-TIFF z-stacks and z-MIPs.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import tifffile

repo = Path.cwd()
src = repo / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from sacdpy.multicolor_zstack import (
    build_batch_plan, load_config, preflight_summary, run_batch, validate_output_pair,
)
print(f'Working folder: {repo}')

## 2. Configuration

The JSON configuration records the raw and output paths, channel-to-frame/camera mapping, accepted metadata wavelengths, and all SACD parameters. Frame boundaries are derived from each TIFF's ONI `laserProgram.steps[].nRepeats` metadata rather than hard-coded in the processing code.

In [ ]:
config_path = repo / 'configs' / 'sacd_multicolor_zstack_20260811.json'
config = load_config(config_path)
print(json.dumps(config, indent=2))

## 3. Read-only discovery and preflight

Run this cell before processing. It validates file names, the z-plane grid, stack dimensions, ONI laser metadata, channel frame totals, camera-split width, pixel size, NA, and unique output paths. It does not create or modify files.

In [ ]:
plan = build_batch_plan(config)
print(json.dumps(preflight_summary(plan), indent=2))
print('\nAccepted FOVs:')
for fov in plan.fovs:
    ranges = [f'{channel.name}: frames {start + 1}-{end}, {channel.camera_half}, PSF {channel.wavelength_nm:g} nm'
              for channel, (start, end) in zip(fov.channels, fov.movies[0].frame_ranges)]
    print(f'+ {fov.relative_fov}: Z={list(fov.z_indices)}, input={fov.movies[0].shape}')
    print('  ' + ' | '.join(ranges))
    print(f'  -> {fov.stack_output.name}')
    print(f'  -> {fov.mip_output.name}')
print('\nExcluded FOVs:')
for item in plan.exclusions:
    print(f"- {item['relative_fov']}: {item['reason']}")

## 4. Run full resumable batch

This is the only processing cell. It processes every discovered FOV into the configured `SACDpy_results` directory, reports each completed channel/z reconstruction, and safely resumes by validating existing OME-TIFF pairs. `_processing/manifest.json`, `run_status.json`, and `run.log` are updated after each FOV.

In [ ]:
results = run_batch(config, config_path)
print(f'Run returned {len(results)} result record(s).')

## 5. Validate and preview the first completed FOV

In [ ]:
preview_result = next(
    item for item in results
    if item['status'] in {'written', 'skipped_existing', 'resumed_manifest'}
)
preview_fov = next(fov for fov in plan.fovs if fov.relative_fov == preview_result['relative_fov'])
validate_output_pair(preview_result['stack_output'], preview_result['mip_output'], preview_fov.channels)
stack = tifffile.imread(preview_result['stack_output'])  # CZYX
mip = tifffile.imread(preview_result['mip_output'])      # CYX
cmaps = {'405': 'Blues', '488': 'Greens', '561': 'Oranges', '647': 'Reds'}
fig, axes = plt.subplots(stack.shape[0], stack.shape[1] + 1,
                         figsize=(3 * (stack.shape[1] + 1), 3 * stack.shape[0]),
                         squeeze=False)
for channel_index, channel in enumerate(preview_fov.channels):
    for z_index in range(stack.shape[1]):
        axes[channel_index, z_index].imshow(stack[channel_index, z_index], cmap=cmaps[channel.name])
        axes[channel_index, z_index].set_title(f'{channel.name}  z{preview_fov.z_indices[z_index]}')
        axes[channel_index, z_index].axis('off')
    axes[channel_index, -1].imshow(mip[channel_index], cmap=cmaps[channel.name])
    axes[channel_index, -1].set_title(f'{channel.name}  SACD MIP')
    axes[channel_index, -1].axis('off')
plt.tight_layout();

## 6. Final summary

In [ ]:
manifest_path = Path(config['output_root']) / '_processing' / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
statuses = {}
for item in manifest.get('fovs', []):
    statuses[item['status']] = statuses.get(item['status'], 0) + 1
print('FOV statuses:', statuses)
print('Output folder:', config['output_root'])